## 1. Setup and Path Configuration

In [ ]:
import json
import cv2
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple
from pycocotools import mask as mask_utils

# ================= CONFIGURAÇÃO DE CAMINHOS =================
unb_server = True

if unb_server:
    BASE_DIR = "/home/antoniovinicius/projects/sandbox_sam3"
    LOG_DIR = Path(f"{BASE_DIR}/logs/olds/ph2_train_seg_500_3")
else:
    BASE_DIR = "/home/avmoura_linux/Documents/unb/sandbox_sam3"
    LOG_DIR = Path(f"{BASE_DIR}/logs/olds/ph2_train_seg_500_3")

CONFIG = {
    "train_stats": LOG_DIR / "train_stats.json",
    "val_stats": LOG_DIR / "val_stats.json",
    "best_stats": LOG_DIR / "best_stats.json",
    
    # Configurações do script de visualização
    "images_dir": Path(f"{BASE_DIR}/datasets/ph2/valid/"),
    "pred_json": Path(f"{LOG_DIR}/dumps/ph2/coco_predictions_segm.json"),
    "gt_json": Path(f"{BASE_DIR}/datasets/ph2/valid/_annotations.coco.json"),
    "score_threshold": 0.35,
    "mask_alpha": 0.45
}

## 2. Data Loading

In [ ]:
def load_jsonl(filepath: Path) -> pd.DataFrame:
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return pd.DataFrame()
        
    data = []
    # Lendo o arquivo jsonl e anexando as linhas
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))
    return pd.DataFrame(data)

# Carregando as métricas de treino, validação e melhores resultados
df_train = load_jsonl(CONFIG["train_stats"])
df_val = load_jsonl(CONFIG["val_stats"])
df_best = load_jsonl(CONFIG["best_stats"])

print(f"Training lines loaded: {len(df_train)}")
print(f"Validation lines loaded: {len(df_val)}")

## 3. Training Loss Analysis

In [ ]:
# Plotando os gráficos de loss (perda) de treinamento
if not df_train.empty:
    plt.figure(figsize=(10, 6))
    
    # Extraindo as colunas de loss disponíveis
    loss_cols = [col for col in df_train.columns if 'loss' in col.lower()]
    
    if 'Losses/train_all_loss' in loss_cols:
        plt.plot(df_train['Trainer/epoch'], df_train['Losses/train_all_loss'], label='Total Loss', color='blue')
        
    if 'Losses/train_all_loss_bbox' in loss_cols:
        plt.plot(df_train['Trainer/epoch'], df_train['Losses/train_all_loss_bbox'], label='BBox Loss', color='green', alpha=0.7)
        
    if 'Losses/train_all_loss_giou' in loss_cols:
        plt.plot(df_train['Trainer/epoch'], df_train['Losses/train_all_loss_giou'], label='GIoU Loss', color='orange', alpha=0.7)

    plt.title('Training Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()
else:
    print("Training data not available.")

## 4. Validation Metrics Analysis

In [ ]:
# Plotando o gráfico das métricas de validação ao longo das épocas
if not df_val.empty:
    plt.figure(figsize=(10, 6))
    
    val_metrics = [col for col in df_val.columns if 'coco_eval_segm_AP' in col or 'coco_eval_bbox_AP' in col]
    
    if 'Meters_train/val_ph2_eval/detection/coco_eval_segm_AP' in val_metrics:
        plt.plot(df_val['Trainer/epoch'], df_val['Meters_train/val_ph2_eval/detection/coco_eval_segm_AP'], label='Segmentation AP', color='purple')
        
    if 'Meters_train/val_ph2_eval/detection/coco_eval_bbox_AP' in val_metrics:
        plt.plot(df_val['Trainer/epoch'], df_val['Meters_train/val_ph2_eval/detection/coco_eval_bbox_AP'], label='BBox AP', color='red')

    plt.title('Validation Average Precision (AP) per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('AP Score')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()
else:
    print("Validation data not available.")

## 5. Validation Results (Last Epoch)

In [ ]:
print("🎯 VALIDATION RESULTS (LAST EPOCH) 🎯")
print("="*50)

# Verificando e extraindo os dados da última época validada
if not df_val.empty:
    last_epoch_data = df_val.iloc[-1]
    epoch_num = last_epoch_data.get('Trainer/epoch', 'N/A')
    
    print(f"Analyzed Epoch: {epoch_num}\n")
    
    metrics_to_print = {}
    for col, value in last_epoch_data.items():
        if "coco_eval" in col and pd.notna(value) and value != -1.0:
            clean_name = col.split('/')[-1].replace('coco_eval_', '')
            metrics_to_print[clean_name] = value

    # Exibindo e formatando os resultados das máscaras e bboxes
    for name, value in sorted(metrics_to_print.items()):
        if 'segm' in name:
            print(f"  🟦 Mask {name.replace('segm_', '')}: \t{value:.4f}")
        elif 'bbox' in name:
            print(f"  🟩 BBox {name.replace('bbox_', '')}: \t{value:.4f}")
else:
    print("No validation data found.")